# Tier 2 — Grid+CV Transformers (Kaggle GPU)

Protocolo: 6 modelos × 6 datasets grandes × 20 seeds = 720 runs.
N_train fixo = 2000 (subsampling estratificado determinístico por seed).

**Antes de rodar:** Settings → Accelerator → GPU T4 x2 (ou P100).

**Resume:** Se a sessão cair, faça download de `tier2_transformers.json` em
Output, suba como dataset Kaggle (Add Data) e descomente `RESUME_PATH` na Célula 5.

In [ ]:
# ── Célula 1: Verifica GPU ──────────────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU não detectada — ative em Settings > Accelerator')

In [ ]:
# ── Célula 2: Clonar repo ───────────────────────────────────────────────────
import os, subprocess

GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'

if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase'], check=True)
else:
    subprocess.run(['git', 'clone', GIT_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
!git log --oneline -3
print(f'Dir: {os.getcwd()}')

In [ ]:
# ── Célula 3: Dependências ──────────────────────────────────────────────────
!pip install -q entmax einops scikit-posthocs openpyxl
# torch, sklearn, numpy, scipy, pandas já presentes no Kaggle
import torch, sklearn, numpy
print(f'torch {torch.__version__} | sklearn {sklearn.__version__} | numpy {numpy.__version__}')

In [ ]:
# ── Célula 4: Datasets Tier 2 ───────────────────────────────────────────────
# Todos os datasets já vêm no repo como parquet — sem download do UCI
import sys; sys.path.insert(0, '.')
from src.data.loaders import DatasetLoader

for ds in ['ADULT', 'BANK', 'CREDIT', 'HIGGS50K', 'SHOPPERS', 'TELCO']:
    X, y, _ = DatasetLoader.load(ds)
    print(f'✓ {ds:<12}  N={len(y):>6}  p={X.shape[1]}')

In [ ]:
# ── Célula 5: Resume de sessão anterior (opcional) ──────────────────────────
# Se quiser continuar de onde parou:
# 1. Suba o JSON anterior como Kaggle dataset (botão Add Data)
# 2. Ajuste o caminho abaixo e descomente as linhas

import shutil, json
from pathlib import Path

OUTPUT_FILE = Path('results/tier2_transformers.json')
OUTPUT_FILE.parent.mkdir(exist_ok=True)

# RESUME_PATH = Path('/kaggle/input/SEU-DATASET/tier2_transformers.json')
# if RESUME_PATH.exists():
#     shutil.copy(RESUME_PATH, OUTPUT_FILE)
#     n = len(json.loads(OUTPUT_FILE.read_text()))
#     print(f'Restaurado: {n} entries')

if OUTPUT_FILE.exists():
    n = len(json.loads(OUTPUT_FILE.read_text()))
    print(f'Iniciando com {n} entries já presentes')
else:
    print('Começando do zero')

In [ ]:
# ── Célula 6: Inspecionar grade ─────────────────────────────────────────────
import sys; sys.path.insert(0, '.')
from src.tuning.grids import GRIDS, grid_size

TRANSFORMER_MODELS = [
    'FTTransformer_softmax', 'FTTransformer_topk',
    'FTTransformer_entmax', 'FTTransformer_sparsemax',
    'SAINTColnorm', 'FTTransformerCURColnorm',
]
TIER2_DATASETS = ['ADULT', 'BANK', 'CREDIT', 'HIGGS50K', 'SHOPPERS', 'TELCO']
N_SEEDS = 30

print(f'{"Modelo":<28}{"Grid":>6}{"Fits/run":>10}')
print('-'*46)
total_runs = 0
for m in TRANSFORMER_MODELS:
    g = grid_size(m)
    runs = len(TIER2_DATASETS) * N_SEEDS
    total_runs += runs
    print(f'{m:<28}{g:>6}{g*5:>10}  ({runs} runs)')
total_fits = sum(grid_size(m) * 5 for m in TRANSFORMER_MODELS) * len(TIER2_DATASETS) * N_SEEDS
print(f'\nTotal runs: {total_runs} | Total fits CV: {total_fits:,}')

In [ ]:
# ── Célula 7: Rodar ─────────────────────────────────────────────────────────
models_str   = ' '.join(TRANSFORMER_MODELS)
datasets_str = ' '.join(TIER2_DATASETS)
seeds_str    = ' '.join(map(str, range(N_SEEDS)))

!python -u scripts/run_tier2_gridcv.py \
    --models {models_str} \
    --datasets {datasets_str} \
    --seeds {seeds_str} \
    --n-train 2000 \
    --output results/tier2_transformers.json \
    2>&1 | tee /kaggle/working/run_tier2.log

In [ ]:
# ── Célula 8: Resumo final ──────────────────────────────────────────────────
import json, statistics as st
from collections import defaultdict
from pathlib import Path

records = json.loads(Path('results/tier2_transformers.json').read_text())
ok = [r for r in records if r.get('status') == 'ok']
total_expected = len(TRANSFORMER_MODELS) * len(TIER2_DATASETS) * N_SEEDS
print(f'Completos: {len(ok)}/{total_expected} ({len(ok)/total_expected*100:.0f}%)')

f1_by_variant = defaultdict(list)
for r in ok:
    f1_by_variant[r['variant']].append(r['test_f1_macro'])

print(f'\n{"Modelo":<30}  F1-macro  n')
for v, vals in sorted(f1_by_variant.items(), key=lambda x: -st.mean(x[1])):
    print(f'{v:<30}  {st.mean(vals):.4f}  {len(vals)}')

# Copiar para output raiz do Kaggle (aparece em Output para download)
import shutil
shutil.copy('results/tier2_transformers.json', '/kaggle/working/tier2_transformers.json')
print('\nArquivo disponível em Output para download.')